# ReproPilot - Unsloth + TRL GRPO Against The Deployed HF Space API

Post-train an instruction model with GRPO where the main reward is fetched from the live ReproPilot OpenEnv Space.

- Live endpoint: `https://riwaj43adz-repro.hf.space`
- Default base model: `unsloth/Qwen2.5-3B-Instruct`
- Algorithm: TRL `GRPOTrainer` through the normal Hugging Face `generate()` path
- Rewards: independent functions for live environment reward, JSON format, action semantics, and anti-idle behavior
- Output: local LoRA adapter, optionally pushed to a Hugging Face model repo

This notebook is designed to be uploaded as a single file to Google Colab. It does not clone or import the ReproPilot repo.


## Phase 1 - Pick The Task

Single-action audit policy learning. The model receives the `/reset` briefing from the deployed ReproPilot Space and must emit exactly one JSON action matching the ReproPilot `AgentAction` schema. The deployed Space scores that action through `/step`; that score is the GRPO learning signal.

This mirrors the previous notebook structure: train from the live API, then inspect generated actions against the same live API.


## Phase 2 - Build The Runtime

Run this cell first in Colab. It uses `!pip install` only and does not upgrade pip.


In [ ]:
%%capture
!pip install -q requests pydantic pandas matplotlib tqdm datasets huggingface_hub
!pip install -q unsloth trl accelerate bitsandbytes


In [ ]:
import os, sys, json, time, random, re, math, pathlib
from typing import Any, Literal

REPROPILOT_ENV_URL = os.environ.get('REPROPILOT_ENV_URL', 'https://riwaj43adz-repro.hf.space').rstrip('/')
MODEL_ID = os.environ.get('REPROPILOT_MODEL_ID', 'unsloth/Qwen2.5-3B-Instruct')
RUN_NAME = os.environ.get('RUN_NAME', 'repropilot-unsloth-grpo-live-api')
HUB_REPO_ID = os.environ.get('HF_MODEL_REPO_ID', 'riwaj43adz/repropilot-qwen2-5-3b-grpo')
PUSH_TO_HUB = os.environ.get('PUSH_TO_HUB', '0') == '1'
SEED = int(os.environ.get('REPROPILOT_SEED', '7'))

OUT = pathlib.Path('/content/repropilot_out') if pathlib.Path('/content').exists() else pathlib.Path('./repropilot_out')
OUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')

print('Endpoint :', REPROPILOT_ENV_URL)
print('Model    :', MODEL_ID)
print('Run name :', RUN_NAME)
print('Output   :', OUT)
print('Hub repo :', HUB_REPO_ID if PUSH_TO_HUB else '(push disabled)')
print('HF token :', 'set' if os.environ.get('HF_TOKEN') else 'missing unless notebook_login is used')


### 2.1 HTTP Client To The Deployed Space

Every environment reward in this notebook comes from this client. `/step` uses OpenEnv's wrapped payload shape: `{"action": {...}}`.


In [ ]:
import requests

class ReproPilotSpace:
    def __init__(self, url: str, timeout: float = 60.0, max_retries: int = 4):
        self.url = url.rstrip('/')
        self.timeout = timeout
        self.max_retries = max_retries
        self.latency_ms: list[float] = []
        self.failures: list[str] = []

    def _post(self, path: str, payload: dict[str, Any]) -> dict[str, Any]:
        last_error: Exception | None = None
        for attempt in range(self.max_retries):
            t0 = time.perf_counter()
            try:
                response = requests.post(f'{self.url}{path}', json=payload, timeout=self.timeout)
                self.latency_ms.append((time.perf_counter() - t0) * 1000)
                if response.status_code >= 400:
                    raise RuntimeError(f'{response.status_code} {response.url}: {response.text[:1200]}')
                return response.json()
            except Exception as exc:
                last_error = exc
                self.failures.append(str(exc))
                time.sleep(min(2.0 * (attempt + 1), 8.0))
        raise RuntimeError(f'Space request failed after retries: {last_error}')

    def reset(self) -> dict[str, Any]:
        return self._post('/reset', {})

    def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return self._post('/step', {'action': action})


env = ReproPilotSpace(REPROPILOT_ENV_URL)
reset_payload = env.reset()
print('reset keys:', sorted(reset_payload.keys()))
print(json.dumps(reset_payload, indent=2)[:1200])


In [ ]:
def unwrap_observation(payload: dict[str, Any]) -> dict[str, Any]:
    inner = payload.get('observation') if isinstance(payload, dict) else None
    if isinstance(inner, dict):
        return inner
    return payload


def briefing_from_reset(payload: dict[str, Any]) -> str:
    obs = unwrap_observation(payload)
    text = obs.get('echoed_message') or obs.get('message') or obs.get('text')
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError(f'Space returned no briefing. keys={list(obs.keys())}')
    return text

briefing = briefing_from_reset(reset_payload)
print(briefing[:2000])


### 2.2 Verifier Sanity Check

Fire representative legal actions once. If all rewards are identical or every step fails, GRPO cannot learn from the verifier.


In [ ]:
LEGAL_ACTION_TYPES = [
    'read_claim',
    'inspect_paper_section',
    'inspect_code_file',
    'inspect_config',
    'inspect_logs',
    'inspect_result_table',
    'inspect_dataset_card',
    'inspect_checkpoint',
    'search_artifacts',
    'compare_claim_to_artifacts',
    'audit_experiment_design',
    'rank_evidence',
    'plan_next_check',
    'run_metric_check',
    'run_split_check',
    'run_leakage_check',
    'run_seed_check',
    'run_ablation_check',
    'run_paper_code_consistency_check',
    'run_reproduction_check',
    'run_dataset_provenance_check',
    'run_hyperparameter_search_check',
    'run_baseline_fairness_check',
    'run_statistical_significance_check',
    'run_implementation_completeness_check',
    'synthesize_findings',
    'mark_inconclusive',
    'submit_verdict',
    'do_nothing',
]
VERDICTS = [
    'SUPPORTED_RESULT_AND_METHOD',
    'PLAUSIBLY_VALIDATED_NOVEL_METHOD',
    'NOT_SUPPORTED_RESULT_MISMATCH',
    'NOT_SUPPORTED_METHOD_INVALID',
    'INCONCLUSIVE_MISSING_ARTIFACTS',
    'INCONCLUSIVE_NEEDS_DOMAIN_REVIEW',
    'NOT_ENOUGH_EVIDENCE',
]
FAILURE_TYPES = [
    'none',
    'metric_mismatch',
    'split_mismatch',
    'data_leakage',
    'cherry_picked_seed',
    'paper_code_mismatch',
    'invalid_ablation',
    'result_mismatch',
    'missing_artifact',
    'ambiguous_method',
    'unsupported_claim',
    'unknown',
]

SMOKE_ACTIONS = [
    {'action_type': 'read_claim', 'explanation': 'Read the target research claim.'},
    {'action_type': 'search_artifacts', 'target_id': 'split', 'explanation': 'Search for split-related artifacts.'},
    {'action_type': 'run_split_check', 'target_id': 'claim_001', 'explanation': 'Verify whether claimed and implemented splits match.'},
    {'action_type': 'run_metric_check', 'target_id': 'claim_001', 'explanation': 'Verify whether claimed and logged metrics match.'},
    {'action_type': 'do_nothing', 'explanation': 'No audit action.'},
]

smoke_rows = []
for action in SMOKE_ACTIONS:
    payload = env.step(action)
    obs = unwrap_observation(payload)
    smoke_rows.append({
        'action_type': action['action_type'],
        'reward': payload.get('reward', obs.get('reward')),
        'done': payload.get('done', obs.get('done')),
        'result': (obs.get('metadata') or {}).get('last_action_result'),
    })

for row in smoke_rows:
    print(row)

rewards = [r['reward'] for r in smoke_rows if isinstance(r.get('reward'), (int, float))]
if len(set(rewards)) <= 1:
    raise RuntimeError(f'Degenerate verifier rewards: {rewards}')


## Phase 3 - Build Rewards

The live environment reward is the main signal. The other rewards are independent guardrails so the trainer can learn the output contract before environment rewards become consistently useful.


In [ ]:
from pydantic import BaseModel, Field

class ReproPilotAction(BaseModel):
    action_type: Literal[
        'read_claim',
        'inspect_paper_section',
        'inspect_code_file',
        'inspect_config',
        'inspect_logs',
        'inspect_result_table',
        'inspect_dataset_card',
        'inspect_checkpoint',
        'search_artifacts',
        'compare_claim_to_artifacts',
        'run_metric_check',
        'run_split_check',
        'run_leakage_check',
        'run_seed_check',
        'run_ablation_check',
        'run_paper_code_consistency_check',
        'run_reproduction_check',
        'synthesize_findings',
        'mark_inconclusive',
        'submit_verdict',
        'do_nothing',
    ] = 'do_nothing'
    target_id: str | None = None
    secondary_id: str | None = None
    verdict: str | None = None
    failure_type: str | None = None
    evidence_ids: list[str] = Field(default_factory=list)
    explanation: str | None = None
    generated_code: str | None = None

_JSON_RE = re.compile(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', re.DOTALL)

def completion_text(c: Any) -> str:
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return str(c[0].get('content', ''))
    return c if isinstance(c, str) else str(c)


def parse_action(text: str) -> tuple[dict[str, Any] | None, str | None]:
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    matches = list(_JSON_RE.finditer(cleaned))
    if not matches:
        return None, 'no_json'
    try:
        obj = json.loads(matches[-1].group(0))
    except json.JSONDecodeError as exc:
        return None, f'bad_json:{exc.msg}'
    try:
        action = ReproPilotAction(**obj).model_dump(exclude_none=True)
    except Exception as exc:
        return None, f'bad_schema:{type(exc).__name__}'
    return action, None


def as_reward(payload: dict[str, Any]) -> float:
    obs = unwrap_observation(payload)
    val = payload.get('reward', obs.get('reward'))
    return float(val or 0.0)


In [ ]:
def format_reward(completions: list[Any], **_: Any) -> list[float]:
    scores = []
    for completion in completions:
        text = completion_text(completion).strip()
        action, error = parse_action(text)
        if error is None and action is not None and text.startswith('{') and text.endswith('}'):
            scores.append(0.25)
        elif error is None:
            scores.append(0.10)
        else:
            scores.append(-0.35)
    return scores


def audit_policy_reward(completions: list[Any], **_: Any) -> list[float]:
    scores = []
    for completion in completions:
        action, error = parse_action(completion_text(completion))
        if action is None or error is not None:
            scores.append(-0.60)
            continue
        at = action.get('action_type')
        score = 0.0
        if at == 'do_nothing':
            score -= 0.35
        elif at == 'read_claim':
            score -= 0.10
        elif at in {'inspect_paper_section', 'inspect_code_file', 'inspect_config', 'inspect_logs', 'inspect_result_table', 'inspect_dataset_card', 'inspect_checkpoint'}:
            score += 0.30 if action.get('target_id') else 0.05
        elif at == 'compare_claim_to_artifacts':
            score += 0.75
        elif at == 'audit_experiment_design':
            score += 0.80
        elif at == 'rank_evidence':
            score += 0.35
        elif at == 'plan_next_check':
            score += 0.30
        elif at == 'search_artifacts':
            query = (action.get('target_id') or action.get('explanation') or '').lower()
            score += 0.25 if any(k in query for k in ['metric', 'split', 'seed', 'artifact', 'result', 'leakage']) else 0.05
        elif at in {
            'run_metric_check',
            'run_split_check',
            'run_leakage_check',
            'run_seed_check',
            'run_ablation_check',
            'run_paper_code_consistency_check',
            'run_reproduction_check',
            'run_dataset_provenance_check',
            'run_hyperparameter_search_check',
            'run_baseline_fairness_check',
            'run_statistical_significance_check',
            'run_implementation_completeness_check',
        }:
            score += 0.55 if action.get('target_id') else 0.30
            if at in {'run_split_check', 'run_metric_check'}:
                score += 0.15
        elif at == 'synthesize_findings':
            score += 0.25
        elif at == 'mark_inconclusive':
            score += 0.05
        elif at == 'submit_verdict':
            has_verdict = action.get('verdict') in VERDICTS
            has_failure = action.get('failure_type') in FAILURE_TYPES
            score += 0.20 if has_verdict and has_failure else -0.10
        if action.get('explanation'):
            score += 0.05
        scores.append(score)
    return scores


def anti_idle_reward(completions: list[Any], **_: Any) -> list[float]:
    scores = []
    for completion in completions:
        action, error = parse_action(completion_text(completion))
        if action is None or error is not None:
            scores.append(-0.25)
        elif action.get('action_type') in {'do_nothing', 'read_claim'}:
            scores.append(-0.15)
        else:
            scores.append(0.10)
    return scores


def env_reward(completions: list[Any], **_: Any) -> list[float]:
    scores = []
    squash = os.environ.get('REPROPILOT_REWARD_SQUASH', '1') == '1'
    scale = float(os.environ.get('REPROPILOT_REWARD_SQUASH_SCALE', '2.5'))
    for completion in completions:
        action, error = parse_action(completion_text(completion))
        if action is None or error is not None:
            scores.append(-1.0)
            continue
        try:
            payload = env.step(action)
            reward = as_reward(payload)
            scores.append(math.tanh(reward / scale) if squash else reward)
        except Exception as exc:
            env.failures.append(str(exc))
            scores.append(-1.0)
    return scores

reward_funcs = [env_reward, format_reward, audit_policy_reward, anti_idle_reward]

probe_actions = [
    '{"action_type":"read_claim","explanation":"Read the target claim."}',
    '{"action_type":"run_split_check","target_id":"claim_001","explanation":"Verify split consistency."}',
    '{"action_type":"search_artifacts","target_id":"metric","explanation":"Search for metric evidence."}',
]
print('env', env_reward(probe_actions))
print('format', format_reward(probe_actions))
print('audit_policy', audit_policy_reward(probe_actions))
print('anti_idle', anti_idle_reward(probe_actions))


### 3.1 Tripwire Callback

Stop early if the trainer collapses to invalid JSON or `do_nothing`.


In [ ]:
from transformers import TrainerCallback

class ReproPilotTripwire(TrainerCallback):
    def __init__(self, *, window: int = 24, max_invalid_rate: float = 0.85, max_idle_rate: float = 0.75):
        self.window = window
        self.max_invalid_rate = max_invalid_rate
        self.max_idle_rate = max_idle_rate
        self.outputs: list[str] = []

    def remember(self, text: str) -> None:
        self.outputs.append(text)
        self.outputs = self.outputs[-self.window:]

    def on_step_end(self, args, state, control, **kwargs):
        if len(self.outputs) < self.window:
            return control
        parsed = [parse_action(x)[0] for x in self.outputs]
        invalid = sum(x is None for x in parsed) / len(parsed)
        idle = sum((x or {}).get('action_type') == 'do_nothing' for x in parsed) / len(parsed)
        if invalid > self.max_invalid_rate or idle > self.max_idle_rate:
            control.should_training_stop = True
            print(f'Tripwire stop: invalid={invalid:.2f} idle={idle:.2f}')
        return control


## Phase 4 - Deploy

Already deployed. Live Space: [`riwaj43adz/repro`](https://huggingface.co/spaces/riwaj43adz/repro). The smoke cells above confirm `/reset` and `/step` are reachable.


## Phase 5 - Train Small

Load the base model in 4-bit with Unsloth, attach LoRA, and run a short GRPO stage against the live Space API.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = int(os.environ.get('REPROPILOT_MAX_SEQ_LENGTH', '2048'))
policy, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False,
)
policy = FastLanguageModel.get_peft_model(
    policy,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
print('loaded', MODEL_ID)


In [ ]:
SYSTEM_PROMPT = f'''
You are ReproPilot, a research-claim audit agent.
You receive a research audit briefing from an OpenEnv environment.
Return exactly one compact JSON object and no Markdown, no prose, no XML tags, no <think> block.

Allowed action_type values: {LEGAL_ACTION_TYPES}
Allowed verdict values: {VERDICTS}
Allowed failure_type values: {FAILURE_TYPES}

Good policy:
- The briefing already contains the claim; do not default to read_claim.
- Prefer inspection, search, plan_next_check, audit_experiment_design, compare_claim_to_artifacts, and deterministic check actions before submit_verdict.
- Use target_id claim_001 for claim-level checks when no better id is visible.
- Use explanations grounded in the briefing.
- Do not invent evidence_ids.
- Avoid do_nothing unless no useful action exists.

Valid examples:
{{"action_type":"run_split_check","target_id":"claim_001","explanation":"Verify whether the claimed test split matches the artifacts."}}
{{"action_type":"audit_experiment_design","target_id":"claim_001","explanation":"Audit split, leakage, hyperparameter search, baselines, and statistical evidence."}}
{{"action_type":"compare_claim_to_artifacts","target_id":"claim_001","explanation":"Compare the claim against available artifacts before choosing a verdict."}}
{{"action_type":"inspect_code_file","target_id":"file_eval","explanation":"Inspect evaluation code before judging the claim."}}
{{"action_type":"search_artifacts","target_id":"metric","explanation":"Search for metric evidence before judging support."}}
'''.strip()


def build_prompt(brief: str) -> list[dict[str, str]]:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': brief + '\n\nReturn one JSON action now.'},
    ]


def render_chat(messages: list[dict[str, str]]) -> str:
    if getattr(tokenizer, 'chat_template', None):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return '\n'.join(f"{m['role'].upper()}: {m['content']}" for m in messages) + '\nASSISTANT:'


def fetch_briefing() -> str:
    return briefing_from_reset(env.reset())


In [ ]:
from datasets import Dataset
from tqdm.auto import tqdm

N_PROMPTS = int(os.environ.get('REPROPILOT_N_PROMPTS', '48'))
briefings = [fetch_briefing() for _ in tqdm(range(N_PROMPTS), desc='fetch briefings')]
train_dataset = Dataset.from_list([{'prompt': render_chat(build_prompt(b))} for b in briefings])
print(train_dataset)
print(train_dataset[0]['prompt'][:1400])


### 5.1 Baselines - Random Action And Frozen Model


In [ ]:
def random_action_json() -> str:
    at = random.choice(LEGAL_ACTION_TYPES)
    action = {'action_type': at, 'explanation': 'Random baseline action.'}
    if at.startswith('run_') or at in {'compare_claim_to_artifacts', 'audit_experiment_design'}:
        action['target_id'] = 'claim_001'
    if at == 'search_artifacts':
        action['target_id'] = random.choice(['metric', 'split', 'seed', 'artifact'])
    if at == 'submit_verdict':
        action.update({'verdict': random.choice(VERDICTS), 'failure_type': random.choice(FAILURE_TYPES), 'evidence_ids': []})
    return json.dumps(action, separators=(',', ':'))

N_EVAL = int(os.environ.get('REPROPILOT_N_EVAL', '8'))
random_rewards = env_reward([random_action_json() for _ in range(N_EVAL)])
print('random rewards:', random_rewards, 'mean=', sum(random_rewards) / len(random_rewards))


### 5.2 Frozen Model JSON Preflight

Before GRPO, sample the base model. If it cannot produce valid JSON at least occasionally, RL will collapse to flat rewards.


In [ ]:
FastLanguageModel.for_inference(policy)

def sample_policy_json(prompt_text: str, *, temperature: float = 0.2) -> tuple[str, dict[str, Any] | None, str | None]:
    inputs = tokenizer(prompt_text, return_tensors='pt').to(policy.device)
    output = policy.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    action, error = parse_action(text)
    return text, action, error

preflight = []
for i in range(min(8, len(train_dataset))):
    text, action, error = sample_policy_json(train_dataset[i]['prompt'])
    preflight.append((text, action, error))

valid = sum(action is not None for _, action, _ in preflight)
print(f'valid_json={valid}/{len(preflight)}')
for text, action, error in preflight[:3]:
    print('\n--- preflight ---')
    print(text[:700])
    print('parsed:', action, 'error:', error)

if valid == 0:
    raise RuntimeError('Base model produced 0 valid JSON actions. Switch MODEL_ID or fix prompt before GRPO.')


### 5.2 Stage B - First GRPO Stage

Purpose: prove the training loop runs, the deployed Space is reachable during training, and rewards move.


In [ ]:
from trl import GRPOConfig, GRPOTrainer
import inspect

stage_logs: dict[str, list[dict[str, Any]]] = {}

def grpo_config(name: str, *, temperature: float, num_generations: int, max_steps: int, lr: float) -> GRPOConfig:
    kwargs = {
        'output_dir': str(OUT / name),
        'run_name': f'{RUN_NAME}-{name}',
        'learning_rate': lr,
        'per_device_train_batch_size': 1,
        'gradient_accumulation_steps': 4,
        'num_generations': num_generations,
        'max_prompt_length': int(os.environ.get('REPROPILOT_MAX_PROMPT_LENGTH', '1600')),
        'max_completion_length': int(os.environ.get('REPROPILOT_MAX_COMPLETION_LENGTH', '96')),
        'max_steps': max_steps,
        'temperature': temperature,
        'logging_steps': 1,
        'save_steps': max_steps,
        'report_to': 'none',
        'log_completions': False,
        'num_completions_to_print': 0,
    }
    accepted = set(inspect.signature(GRPOConfig).parameters)
    return GRPOConfig(**{k: v for k, v in kwargs.items() if k in accepted})


def run_stage(name: str, **kw: Any):
    cfg = grpo_config(name, **kw)
    trainer = GRPOTrainer(
        model=policy,
        processing_class=tokenizer,
        reward_funcs=reward_funcs,
        args=cfg,
        train_dataset=train_dataset,
    )
    result = trainer.train()
    stage_logs[name] = trainer.state.log_history
    print(name, result.metrics)
    return trainer

trainer_b = run_stage('B', temperature=0.45, num_generations=4, max_steps=30, lr=5e-6)


## Phase 6 - Inspect For Hacking

Sample post-training completions, parse them, hit the live Space, and print completion/action/reward. Do this before trusting mean reward.


In [ ]:
FastLanguageModel.for_inference(policy)

for i in range(6):
    prompt_text = train_dataset[i % len(train_dataset)]['prompt']
    inputs = tokenizer(prompt_text, return_tensors='pt').to(policy.device)
    output = policy.generate(
        **inputs,
        max_new_tokens=96,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    action, error = parse_action(text)
    reward = env_reward([text])[0]
    print('\n--- sample', i, '---')
    print(text)
    print('parsed:', action, 'error:', error, 'reward:', reward)


## Phase 7 - Add Curriculum

The deployed Space controls the environment. We harden behavior by lowering temperature and learning rate after the first stage.


In [ ]:
trainer_c = run_stage('C', temperature=0.35, num_generations=4, max_steps=40, lr=3e-6)
trainer_d = run_stage('D', temperature=0.25, num_generations=4, max_steps=25, lr=2e-6)


## Phase 8 - Plot Rewards


In [ ]:
import matplotlib.pyplot as plt

all_logs = []
for stage, logs in stage_logs.items():
    for row in logs:
        if 'reward' in row:
            all_logs.append({'stage': stage, 'step': row.get('step', len(all_logs)), **row})

if all_logs:
    xs = list(range(len(all_logs)))
    plt.figure(figsize=(10, 4))
    plt.plot(xs, [r.get('reward', 0.0) for r in all_logs], marker='o', label='total')
    for key in ['rewards / env_reward / mean', 'rewards / format_reward / mean', 'rewards / audit_policy_reward / mean', 'rewards / anti_idle_reward / mean']:
        vals = [r.get(key) for r in all_logs]
        if any(v is not None for v in vals):
            plt.plot(xs, [0.0 if v is None else v for v in vals], marker='.', label=key.split('/')[1].strip())
    plt.title('ReproPilot GRPO reward channels vs live HF Space')
    plt.xlabel('logged step')
    plt.ylabel('reward')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 3))
    plt.plot(xs, [r.get('reward_std', 0.0) for r in all_logs], marker='o')
    plt.title('Reward std inside GRPO groups')
    plt.xlabel('logged step')
    plt.ylabel('reward_std')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('No reward logs found. Inspect trainer.state.log_history manually.')

print('space calls:', len(env.latency_ms), 'failures:', len(env.failures))


## Phase 9 - Save And Push

This saves the LoRA adapter locally. Set `PUSH_TO_HUB=1` before the notebook run, or run the login cell here and push manually.


In [ ]:
ADAPTER_DIR = OUT / 'final_adapter'
policy.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved adapter:', ADAPTER_DIR)

if PUSH_TO_HUB:
    from huggingface_hub import login, notebook_login
    token = os.environ.get('HF_TOKEN')
    if token:
        login(token=token)
    else:
        notebook_login()
    policy.push_to_hub(HUB_REPO_ID, commit_message=f'ReproPilot GRPO adapter ({RUN_NAME})')
    tokenizer.push_to_hub(HUB_REPO_ID)
    print('pushed:', HUB_REPO_ID)
else:
    print('push disabled; set PUSH_TO_HUB=1 and HF_MODEL_REPO_ID to upload')


## Phase 10 - Run Manifest


In [ ]:
manifest = {
    'run_name': RUN_NAME,
    'env_url': REPROPILOT_ENV_URL,
    'model_id': MODEL_ID,
    'hub_repo_id': HUB_REPO_ID if PUSH_TO_HUB else None,
    'n_space_calls': len(env.latency_ms),
    'mean_space_latency_ms': round(sum(env.latency_ms) / max(len(env.latency_ms), 1), 1),
    'space_failures_tail': env.failures[-10:],
    'out': str(OUT),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
